# INSTALL

In [1]:
!pip install typhoon-ocr pdf2image Pillow

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 13.2 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [pypdf]
   ----- ---------------------------------- 1/8 [pypdf]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   ------------------------------ --------- 6/8 [openai]
   

In [2]:
!apt-get install -y poppler-utils

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!pip install -U google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 11.9 MB/s  0:00:00
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ------------------ --------------------- 2.4/5.0 MB 13.9 MB/s eta 0:00:01
   --------------------------------- ------ 4.2/5.0 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------- 5.0/5.0 MB 9.6 MB/s  0:00:00
   ----------

In [10]:
!pip install python-dotenv

# IMPORT

In [25]:
import os
import json
import re
import requests
# import dashscope
import shutil
from pdf2image import convert_from_path
from typhoon_ocr import ocr_document
import google.generativeai as genai
from dotenv import load_dotenv

# Pipeline

In [26]:
class Extractor:
    #  Initialize
    def __init__(self, typhoon_key: str, gemini_key: str):
        """Initializes the OCR engine with the provided API key."""
        # OCR
        os.environ["TYPHOON_OCR_API_KEY"] = typhoon_key
        # Parse by Gemini
        genai.configure(api_key=gemini_key)
        self.model = genai.GenerativeModel('gemini-2.5-flash')
        # Parse by Gemini
        # dashscope.api_key = qwen_key
        # self.model_name = "qwen-plus"

    VALID_PARTIES = """
    ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เครือข่ายชาวนาแห่งประเทศไทย, เพื่อไทย, 
    ชาติพัฒนา, ชาติไทยพัฒนา, อนาคตไทย, ภูมิใจไทย, สังคมประชาธิปไตยไทย, รักชาติ, 
    ประชาธิปไตยใหม่, พลังบูรพา, ครูไทยเพื่อประชาชน, พลังท้องถิ่นไท, ประชาชน, 
    ไทยก้าวใหม่, เสรีรวมไทย, รักษ์ธรรม, พลังประชาธิปไตย, พลังสุราษฎร์, พลังไทยรักชาติ, 
    เพื่อชีวิตใหม่, ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม, 
    รวมพลัง, ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, พร้อมพัฒนา, ประชาชาติ, แผ่นดินธรรม, 
    คลองไทย, พลังประชารัฐ, เศรษฐกิจใหม่, พลังสังคม, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, 
    กรีน, วิชชั่นใหม่, พลวัต, กล้าธรรม, ไทยรวมไทย, กล้า, ฟิวชัน, พลังสังคมใหม่, 
    ไทยสร้างไทย, รวมไทยสร้างชาติ, มิติใหม่, ไทยสมาร์ท, ไทยภักดี, ไทยพิทักษ์ธรรม, 
    ไทยชนะ, ไทรวมพลัง, ราษฎร์วิถี, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, 
    ไทยก้าวหน้า, ตะวันใหม่, พร้อม, รวมใจไทย, สัมมาธิปไตย, รักภูเก็ต, ประชาอาสาชาติ, 
    ไทยทรัพย์ทวี, รวมพลังประชาชน, อนาคตไกล, ยางพาราไทย, เพื่อบ้านเมือง
    """
    
    #  Image to dict
    def extract_pdf(self, pdf_path: str) -> str:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"The file {pdf_path} was not found.")
        
        print(f"Converting PDF with Typhoon OCR: {pdf_path}")

        # raw_markdown = ocr_document(pdf_path)
        temp_dir = "temp_pages"
        if not os.path.exists(temp_dir):
            os.makedirs(temp_dir)
            
        full_markdown_text = ""
        
        try:
            pages = convert_from_path(pdf_path)
            
            for i, page in enumerate(pages):
                page_path = os.path.join(temp_dir, f"page_{i+1}.jpg")
                page.save(page_path, "JPEG")
                
                print(f"--- OCRing Page {i+1}/{len(pages)} ---")
                page_markdown = ocr_document(page_path)
                full_markdown_text += f"\n--- PAGE {i+1} ---\n" + page_markdown
            
            print("Sending merged text to Gemini for final parsing...")
            return self._parse_markdown_byGemini(full_markdown_text)
            
        finally:
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)

        return self._parse_markdown_byGemini(raw_markdown)
    
    # Text to Dict
    def _parse_markdown_byGemini(self, full_text: str) -> str:
        # PROMPT
        prompt = f"""
        Extract the election results from the following Thai OCR text into a structured JSON format.
        
        Requirements:
        1. Fix any OCR typos in province, district, or party names.
        2. Convert all numbers (including Thai digits) to standard integers.
        3. Include a 'metadata' section (province, district, unit), a 'summary' section (ballot counts), 
           and a 'results' list (party number, name, and votes).
        4. **PARTY NAME VALIDATION**: Compare the party name from OCR with the following valid list. 
           If the OCR name is misspelled or slightly different, you MUST change it to the correct name from this list:
           {self.VALID_PARTIES}
        5. Check the score: look for numbers and Thai text in parentheses (). 
           If the main number is messy, convert the Thai text description into an integer.
        OCR Text:
        {full_text}
        """

        response = self.model.generate_content(
            prompt,
            generation_config={"response_mime_type": "application/json"}
        )
        return response.text
    
    # def _parse_markdown_byQwen(self, full_text: str) -> str:
    #     prompt = f"""
    #     Extract the election results from the following Thai OCR text into a structured JSON format.
        
    #     Requirements:
    #     1. Fix any OCR typos in province, district, or party names.
    #     2. Convert all numbers (including Thai digits) to standard integers.
    #     3. Include a 'metadata' section (province, district, unit), a 'summary' section (ballot counts), 
    #        and a 'results' list (party number, name, and votes).
    #     4. Check the score: look for numbers and Thai text in parentheses (). 
    #        If the main number is messy, convert the Thai text description into an integer.

    #     OCR Text:
    #     {full_text}
    #     """

    #     response = dashscope.Generation.call(
    #         model=self.model_name,
    #         prompt=prompt,
    #         result_format='message' # Returns data in a format similar to OpenAI
    #     )

    #     if response.status_code == 200:
    #         content = response.output.choices[0].message.content
    #         return self._clean_json_string(content)
    #     else:
    #         raise Exception(f"Qwen Error: {response.code} - {response.message}")
        
    def _clean_json_string(self, content: str) -> str:
        """Helper to remove markdown code blocks if Qwen includes them."""
        content = content.strip()
        if content.startswith("```json"):
            content = content[7:]
        if content.endswith("```"):
            content = content[:-3]
        return content.strip()



        

In [27]:
if __name__ == "__main__":

    load_dotenv()
    
    TYPHOON_KEY = os.getenv("TYPHOON_KEY")
    GEMINI_KEY = os.getenv("GEMINI_KEY")
    
    extractor = Extractor(TYPHOON_KEY, GEMINI_KEY)
    
    try:
        json_output = extractor.extract_pdf("doc1.pdf")
        print("--- Extraction Result ---")
        print(json_output)
        
        # Save locally
        with open("output_gemini.json", "w", encoding="utf-8") as f:
            f.write(json_output)
            
    except Exception as e:
        print(f"An error occurred: {e}")

Converting PDF with Typhoon OCR: doc1.pdf
--- OCRing Page 1/4 ---
--- OCRing Page 2/4 ---
--- OCRing Page 3/4 ---
--- OCRing Page 4/4 ---
Sending merged text to Gemini for final parsing...
--- Extraction Result ---
{
  "metadata": {
    "province": "อุบลราชธานี",
    "district": "เมือง",
    "sub_district": "ไม่น้อย",
    "unit_number": 1,
    "village_number": 1,
    "election_district": 2
  },
  "summary": {
    "eligible_voters_list": 705,
    "eligible_voters_present": 513,
    "ballots_allocated": 700,
    "ballots_used": 513,
    "good_ballots": 468,
    "bad_ballots": 20,
    "no_party_choice_ballots": 25,
    "ballots_remaining": 187
  },
  "results": [
    {
      "party_number": 1,
      "party_name": "ไทยทรัพย์ทวี",
      "votes": 6
    },
    {
      "party_number": 2,
      "party_name": "เพื่อชาติไทย",
      "votes": 7
    },
    {
      "party_number": 3,
      "party_name": "ใหม่",
      "votes": 4
    },
    {
      "party_number": 4,
      "party_name": "มิติใหม่",
  